In [19]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import shap
import os
import warnings

warnings.filterwarnings('ignore')

In [20]:
# =====================================================================
# 0. SETUP OUTPUT DIRECTORIES
# =====================================================================
charts_dir = "evaluation_charts"
models_dir = "models"
projections_dir = "projections"

os.makedirs(charts_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(projections_dir, exist_ok=True)

In [21]:
# =====================================================================
# 1. LOAD DATA 
# =====================================================================
print("📥 Loading and merging relational datasets...")
try:
    train_folder = 'dataset/train'
    test_folder = 'dataset/test'
    
    train_aqi_df = pd.read_csv(f'{train_folder}/air_quality_train.csv')
    train_flood_df = pd.read_csv(f'{train_folder}/flood_risk_train.csv')
    train_housing_df = pd.read_csv(f'{train_folder}/housing_data_train.csv')
    
    test_aqi_df = pd.read_csv(f'{test_folder}/air_quality_test.csv')
    test_flood_df = pd.read_csv(f'{test_folder}/flood_risk_test.csv')
    test_housing_df = pd.read_csv(f'{test_folder}/housing_data_test.csv')

except FileNotFoundError:
    print(f"❌ Error: Could not find the CSV files.")
    exit()

📥 Loading and merging relational datasets...


In [22]:
# =====================================================================
# 2. PREPROCESSING (STRICT INDEPENDENT ARCHITECTURE)
# =====================================================================
train_merged = train_housing_df.merge(train_aqi_df, on=['Record_ID', 'Year', 'Barangay']).merge(train_flood_df, on=['Record_ID', 'Year', 'Barangay'])
test_merged = test_housing_df.merge(test_aqi_df, on=['Record_ID', 'Year', 'Barangay']).merge(test_flood_df, on=['Record_ID', 'Year', 'Barangay'])

rename_map = {
    'Year': 'year', 'Barangay': 'barangay', 'Proximity_km': 'proximity_km', 
    'AQI': 'air_quality_aqi', 'Flood_Risk': 'flood_risk', 'Price_PHP': 'housing_price_php'
}
train_merged.rename(columns=rename_map, inplace=True)
test_merged.rename(columns=rename_map, inplace=True)

y_train_price = train_merged['housing_price_php']
y_train_aqi = train_merged['air_quality_aqi']
y_train_flood = train_merged['flood_risk']

y_test_price = test_merged['housing_price_php']
y_test_aqi = test_merged['air_quality_aqi']
y_test_flood = test_merged['flood_risk']

X_train_base = train_merged.drop(columns=['Record_ID', 'housing_price_php', 'air_quality_aqi', 'flood_risk'])
X_test_base = test_merged.drop(columns=['Record_ID', 'housing_price_php', 'air_quality_aqi', 'flood_risk'])

X_train_encoded = pd.get_dummies(X_train_base, columns=['barangay'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test_base, columns=['barangay'], drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

spatial_cols = [col for col in X_train_encoded.columns if 'barangay_' in col]

# MASTER FEATURE SET: Only Time and Location! 
features = ['year', 'proximity_km'] + spatial_cols

X_train_final = X_train_encoded[features]
X_test_final = X_test_encoded[features]

In [23]:
# =====================================================================
# 3. TRAIN AND TUNE ALL MODELS (FULL GRIDSEARCHCV)
# =====================================================================
# Define a slightly smaller grid for environment to save a little time
param_grid_env = {'max_depth': [3, 4], 'learning_rate': [0.01, 0.05], 'n_estimators': [100, 150]}

print("🧠 Tuning and Training Model A: Air Quality (AQI) Predictor...")
grid_search_aqi = GridSearchCV(xgb.XGBRegressor(random_state=42), param_grid_env, cv=3, scoring='neg_mean_absolute_error')
grid_search_aqi.fit(X_train_final, y_train_aqi)
model_aqi = grid_search_aqi.best_estimator_

# Track learning curve for AQI
model_aqi.fit(X_train_final, y_train_aqi, eval_set=[(X_train_final, y_train_aqi), (X_test_final, y_test_aqi)], verbose=False)
results_aqi_lc = model_aqi.evals_result()


print("🧠 Tuning and Training Model B: Flood Risk Predictor...")
grid_search_flood = GridSearchCV(xgb.XGBRegressor(random_state=42), param_grid_env, cv=3, scoring='neg_mean_absolute_error')
grid_search_flood.fit(X_train_final, y_train_flood)
model_flood = grid_search_flood.best_estimator_

# Track learning curve for Flood
model_flood.fit(X_train_final, y_train_flood, eval_set=[(X_train_final, y_train_flood), (X_test_final, y_test_flood)], verbose=False)
results_flood_lc = model_flood.evals_result()


print("🧠 Tuning and Training Model C: Housing Value Predictor...")
param_grid_price = {'max_depth': [3, 4, 5], 'learning_rate': [0.01, 0.05, 0.1], 'n_estimators': [200, 300]}
grid_search_price = GridSearchCV(xgb.XGBRegressor(objective='reg:squarederror', random_state=42), param_grid_price, cv=3, scoring='neg_mean_absolute_error')
grid_search_price.fit(X_train_final, y_train_price)
model_price = grid_search_price.best_estimator_

# Track learning curve for Price
model_price.fit(X_train_final, y_train_price, eval_set=[(X_train_final, y_train_price), (X_test_final, y_test_price)], verbose=False)
results_price_lc = model_price.evals_result()

🧠 Tuning and Training Model A: Air Quality (AQI) Predictor...
🧠 Tuning and Training Model B: Flood Risk Predictor...
🧠 Tuning and Training Model C: Housing Value Predictor...


In [24]:
# =====================================================================
# 4. EVALUATION, METRICS, AND CHARTS
# =====================================================================
print("📊 Generating Scientific Evaluation Charts & Metrics...")

# --- PREDICTIONS & METRICS ---
train_price_preds = model_price.predict(X_train_final)
test_price_preds = model_price.predict(X_test_final)
price_train_mae = mean_absolute_error(y_train_price, train_price_preds)
price_test_mae = mean_absolute_error(y_test_price, test_price_preds)
price_test_rmse = np.sqrt(mean_squared_error(y_test_price, test_price_preds))
price_mape = mean_absolute_percentage_error(y_test_price, test_price_preds)
price_accuracy = (1 - price_mape) * 100

aqi_train_preds = model_aqi.predict(X_train_final)
aqi_test_preds = model_aqi.predict(X_test_final)
aqi_train_mae = mean_absolute_error(y_train_aqi, aqi_train_preds)
aqi_test_mae = mean_absolute_error(y_test_aqi, aqi_test_preds)
aqi_test_rmse = np.sqrt(mean_squared_error(y_test_aqi, aqi_test_preds))
aqi_mape = mean_absolute_percentage_error(y_test_aqi, aqi_test_preds)

flood_train_preds = model_flood.predict(X_train_final)
flood_test_preds = model_flood.predict(X_test_final)
flood_train_mae = mean_absolute_error(y_train_flood, flood_train_preds)
flood_test_mae = mean_absolute_error(y_test_flood, flood_test_preds)
flood_test_rmse = np.sqrt(mean_squared_error(y_test_flood, flood_test_preds))

# --- TERMINAL REPORT CARDS ---
print("\n" + "="*50)
print("FINAL MODEL VALIDATION REPORT")
print("="*50)

print("\n🏠 Model C: Housing Price Predictor")
print(f"  MAE:  ₱{price_test_mae:,.2f}")
print(f"  RMSE: ₱{price_test_rmse:,.2f}")
print(f"  MAPE: {(price_mape * 100):.2f}%")
print(f"  OVERALL ACCURACY: {price_accuracy:.2f}%")

print("\n🌬️ Model A: Air Quality (AQI) Predictor")
print(f"  MAE:  {aqi_test_mae:.2f} AQI points")
print(f"  RMSE: {aqi_test_rmse:.2f} AQI points")
print(f"  MAPE: {(aqi_mape * 100):.2f}%")

print("\n🌊 Model B: Flood Risk Predictor")
print(f"  MAE:  {flood_test_mae:.2f} Hazard levels")
print(f"  RMSE: {flood_test_rmse:.2f} Hazard levels")
print("  *(Note: Flood MAPE excluded due to safe division-by-zero handling)*\n")

# --- GENERATE CHARTS ---
# Chart 1: Tuning Impact (Price)
results = pd.DataFrame(grid_search_price.cv_results_)
plt.figure(figsize=(8, 5))
plt.plot(range(len(results)), -results['mean_test_score'], marker='o', color='purple')
plt.title("Impact of Hyperparameter Tuning on Error Rate")
plt.xlabel("Tuning Iteration")
plt.ylabel("Mean Absolute Error (Lower is Better)")
plt.grid(True)
plt.savefig(f'{charts_dir}/chart_1_tuning_impact.png', bbox_inches='tight')
plt.close()

# Chart 2: Learning Curves (All 3 Models)
plt.figure(figsize=(10, 5))
plt.plot(results_price_lc['validation_0']['rmse'], label='Train Error')
plt.plot(results_price_lc['validation_1']['rmse'], label='Test Error')
plt.title("Price Model Learning Curve")
plt.legend()
plt.grid(True)
plt.savefig(f'{charts_dir}/chart_2_price_learning_curve.png', bbox_inches='tight')
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(results_aqi_lc['validation_0']['rmse'], label='Train Error', color='green')
plt.plot(results_aqi_lc['validation_1']['rmse'], label='Test Error', color='lime')
plt.title("AQI Model Learning Curve")
plt.legend()
plt.grid(True)
plt.savefig(f'{charts_dir}/chart_2a_aqi_learning_curve.png', bbox_inches='tight')
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(results_flood_lc['validation_0']['rmse'], label='Train Error', color='teal')
plt.plot(results_flood_lc['validation_1']['rmse'], label='Test Error', color='cyan')
plt.title("Flood Model Learning Curve")
plt.legend()
plt.grid(True)
plt.savefig(f'{charts_dir}/chart_2b_flood_learning_curve.png', bbox_inches='tight')
plt.close()

# Chart 3: Evaluation Metrics Bar Charts
plt.figure(figsize=(8, 5))
bars = plt.bar(['Train MAE', 'Test MAE'], [price_train_mae, price_test_mae], color=['blue', 'lightblue'])
plt.ylim(0, max(price_train_mae, price_test_mae) * 1.2) 
plt.title("Housing Price Evaluation: Training vs Testing")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(price_train_mae, price_test_mae) * 0.03), 
             f"₱{yval:,.0f}", ha='center', fontweight='bold')
plt.savefig(f'{charts_dir}/chart_3_price_evaluation.png', bbox_inches='tight')
plt.close()

# Chart 4, 7, 8: Feature Importance
plt.figure(figsize=(10, 8))
xgb.plot_importance(model_price, max_num_features=10, importance_type='weight', title='Price Predictor: Feature Importance')
plt.savefig(f'{charts_dir}/chart_4_price_importance.png', bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 6))
xgb.plot_importance(model_aqi, max_num_features=10, importance_type='weight', title='AQI Predictor: Feature Importance')
plt.savefig(f'{charts_dir}/chart_7_aqi_importance.png', bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 6))
xgb.plot_importance(model_flood, max_num_features=10, importance_type='weight', title='Flood Predictor: Feature Importance')
plt.savefig(f'{charts_dir}/chart_8_flood_importance.png', bbox_inches='tight')
plt.close()

# Chart 5: SHAP Explainability (Proves independence)
X_test_shap = X_test_final.astype(float)
explainer = shap.Explainer(model_price, X_test_shap)
shap_values = explainer(X_test_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_shap, show=False)
plt.title("SHAP Feature Summary (Independent Architecture)")
plt.savefig(f'{charts_dir}/chart_5_shap_summary.png', bbox_inches='tight')
plt.close()


📊 Generating Scientific Evaluation Charts & Metrics...

FINAL MODEL VALIDATION REPORT

🏠 Model C: Housing Price Predictor
  MAE:  ₱112,749.87
  RMSE: ₱136,226.36
  MAPE: 5.92%
  OVERALL ACCURACY: 94.08%

🌬️ Model A: Air Quality (AQI) Predictor
  MAE:  2.50 AQI points
  RMSE: 3.12 AQI points
  MAPE: 3.44%

🌊 Model B: Flood Risk Predictor
  MAE:  0.32 Hazard levels
  RMSE: 0.41 Hazard levels
  *(Note: Flood MAPE excluded due to safe division-by-zero handling)*



<Figure size 1000x800 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

In [25]:
# --- Chart 3A: AQI Evaluation Metrics Bar Chart ---
plt.figure(figsize=(8, 5))
bars = plt.bar(['Train MAE', 'Test MAE'], [aqi_train_mae, aqi_test_mae], color=['green', 'lightgreen'])
plt.ylim(0, max(aqi_train_mae, aqi_test_mae) * 1.2) 
plt.title("AQI Evaluation: Training vs Testing")
plt.ylabel("Error in AQI Points")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(aqi_train_mae, aqi_test_mae) * 0.03), 
             f"{yval:.2f}", ha='center', fontweight='bold')
plt.savefig(f'{charts_dir}/chart_3a_aqi_evaluation.png', bbox_inches='tight')
plt.close()

# --- Chart 3B: Flood Risk Evaluation Metrics Bar Chart ---
plt.figure(figsize=(8, 5))
bars = plt.bar(['Train MAE', 'Test MAE'], [flood_train_mae, flood_test_mae], color=['teal', 'cyan'])
plt.ylim(0, max(flood_train_mae, flood_test_mae) * 1.2) 
plt.title("Flood Risk Evaluation: Training vs Testing")
plt.ylabel("Error in Hazard Levels")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(flood_train_mae, flood_test_mae) * 0.03), 
             f"{yval:.2f}", ha='center', fontweight='bold')
plt.savefig(f'{charts_dir}/chart_3b_flood_evaluation.png', bbox_inches='tight')
plt.close()

In [26]:
# =====================================================================
# 5. SAVE FINAL JSON MODELS
# =====================================================================
print("💾 Saving independent .json models for FastAPI backend...")
model_aqi.save_model(f"{models_dir}/model_aqi.json")
model_flood.save_model(f"{models_dir}/model_flood.json")
model_price.save_model(f"{models_dir}/model_price.json")

💾 Saving independent .json models for FastAPI backend...


In [27]:
# =====================================================================
# PHASE 6: THE DASHBOARD SIMULATOR (FORECASTING TO 2045)
# =====================================================================
print("\n" + "="*120)
print(f"FINAL DASHBOARD PROJECTIONS: INDEPENDENT ARCHITECTURE (Base Accuracy: {price_accuracy:.2f}%)")
print("="*120)

proximity_map = {
    'Amaya': 1.8, 'Bagtas': 6.9, 'Biga': 4.8, 'Biwas': 2.1, 'Bucal': 1.9, 
    'Bunga': 1.7, 'Calibuyo': 6.8, 'Capipisa': 9.0, 'Daang Amaya': 0.8,
    'Halayhay': 6.1, 'Julugan': 3.0, 'Lambingan': 10.4, 'Mulawin': 1.5, 
    'Paradahan': 8.9, 'Poblacion': 0.6, 'Punta': 4.4, 'Sahud Ulan': 5.9, 
    'Sanja Mayor': 2.5, 'Santol': 3.0, 'Tanauan': 17.6, 'Tres Cruses': 9.6
}

GLOBAL_MIN_PRICE = 1500000
GLOBAL_MAX_PRICE = 50000000

def calculate_price_score(price):
    ratio = (price - GLOBAL_MIN_PRICE) / (GLOBAL_MAX_PRICE - GLOBAL_MIN_PRICE)
    return max(0, min(100, 100 - (ratio * 100)))

projection_results = []

def get_future_projections(target_barangay, current_year=2025):
    timeframes = [0, 5, 10, 15, 20]
    prox = proximity_map.get(target_barangay, 5.0)
    
    print(f"\nLocation: {target_barangay.upper()} (Distance: {prox}km)")
    print(f"{'Year':<6} | {'Timeline':<12} | {'AQI':<6} | {'Flood':<6} | {'Price (₱)':<14} | {'Housing Value (0-100)':<21} | {'Confidence'}")
    print("-" * 100)
    
    for offset in timeframes:
        target_year = current_year + offset
        timeline_label = "Current" if offset == 0 else f"+{offset} Years"
        xgboost_year = min(target_year, 2025)
        years_into_future = max(0, target_year - 2025)
        
        # 1. Prepare Base Input (SAME FOR ALL 3 MODELS)
        input_dict = {'year': xgboost_year, 'proximity_km': prox}
        for col in spatial_cols:
            input_dict[col] = 1 if f"barangay_{target_barangay}" == col else 0
            
        df_input = pd.DataFrame([input_dict])[features]
        
        # 2. Predict Independently
        base_aqi = model_aqi.predict(df_input)[0]
        base_flood = model_flood.predict(df_input)[0]
        base_price = model_price.predict(df_input)[0]
        
        # 3. Apply Deterministic Creep
        predicted_aqi = base_aqi + (years_into_future * 0.5)
        predicted_flood = base_flood + (years_into_future * 0.02)
        predicted_price = base_price * ((1 + 0.06) ** years_into_future)
        
        # 4. Normalize
        aqi_score = max(0, min(100, 100 - ((predicted_aqi / 150.0) * 100)))
        flood_score = max(0, min(100, 100 - ((predicted_flood / 5.0) * 100)))
        prox_score = max(0, min(100, 100 - (prox * 10)))
        price_score = calculate_price_score(predicted_price)
        
        housing_value = (price_score * 0.40) + (flood_score * 0.25) + (prox_score * 0.20) + (aqi_score * 0.15)
        degraded_confidence = price_accuracy - (offset * 0.4) 
        
        price_str = f"₱{predicted_price / 1000000:.2f}M" 
        print(f"{target_year:<6} | {timeline_label:<12} | {predicted_aqi:<6.1f} | {predicted_flood:<6.2f} | {price_str:<14} | {housing_value:>6.1f} / 100       | {degraded_confidence:.1f}%")
        
        projection_results.append({
            'Barangay': target_barangay, 'Year': target_year, 'Timeline': timeline_label,
            'Predicted_AQI': round(predicted_aqi, 1), 'Predicted_Flood': round(predicted_flood, 2),
            'Predicted_Price_PHP': round(predicted_price, 2), 'Computed_Housing_Value': round(housing_value, 1),
            'Confidence': f"{degraded_confidence:.1f}%"
        })

for brgy in sorted(list(proximity_map.keys())):
    get_future_projections(brgy)

pd.DataFrame(projection_results).to_csv(f'{projections_dir}/final_dashboard_projections.csv', index=False)
print(f"\n✅ SUCCESS! Master Pipeline complete. Check your evaluation_charts/ folder!")


FINAL DASHBOARD PROJECTIONS: INDEPENDENT ARCHITECTURE (Base Accuracy: 94.08%)

Location: AMAYA (Distance: 1.8km)
Year   | Timeline     | AQI    | Flood  | Price (₱)      | Housing Value (0-100) | Confidence
----------------------------------------------------------------------------------------------------
2025   | Current      | 69.7   | 3.07   | ₱2.49M         |   73.3 / 100       | 94.1%
2030   | +5 Years     | 72.2   | 3.17   | ₱3.33M         |   71.8 / 100       | 92.1%
2035   | +10 Years    | 74.7   | 3.27   | ₱4.45M         |   70.1 / 100       | 90.1%
2040   | +15 Years    | 77.2   | 3.37   | ₱5.96M         |   68.1 / 100       | 88.1%
2045   | +20 Years    | 79.7   | 3.47   | ₱7.98M         |   65.7 / 100       | 86.1%

Location: BAGTAS (Distance: 6.9km)
Year   | Timeline     | AQI    | Flood  | Price (₱)      | Housing Value (0-100) | Confidence
----------------------------------------------------------------------------------------------------
2025   | Current      | 73.1  